# Six-basin, three-run MOI mass-conservation experiment

This notebook follows yesterday's targeted-basin workflow, but fixes the basin set to `7429`, `7426`, `6424`, `7521`, `7424`, and `6412` and compares three controlled runs:

1. **run1** — unconstrained MOI, with no gage constraints.
2. **run2** — constrained MOI using only the rows marked `calibration` in `CalValSeparation_basin_stratified_v2.csv`.
3. **run3** — constrained MOI after assigning every listed gage in the six basins to `calibration`.

The experiment keeps exact in-memory MetroMan `qbar_reachScale` and `qbar_basinScale` values in CSV files. It runs **Mean-only**, skips `q33` and final flow-law parameter estimation, and does **not** write per-reach NetCDF or SWORD output. For each basin, a 2×3 map colors the SWORD river centerlines by qbar and marks non-conserving junctions with red crosses. Junction mass closure is evaluated at both scales and the non-conserving fraction is printed.


In [ ]:
# Cell 1 — experiment configuration
from pathlib import Path
import importlib
import os
import sys
import warnings

TARGET_BASINS = ["7429", "7426", "6424", "7521", "7424", "6412"]

RUN_ROOT = Path(
    "/nas/cee-water/cjgleason/Yushan/Confluence_Aug/"
    "confluence_global_v17c_gagecorr/global_v17c_gagecorr_mnt"
)
RUN_TAG = "six_basin_three_run_mass_conservation_v1"
RESULT_ROOT = RUN_ROOT / "experiments" / RUN_TAG

RUN_EXPERIMENTS = True
REUSE_EXISTING = True
CONTINUE_ON_ERROR = True
VERBOSE = False
MEAN_ONLY = True
PARAMETER_OVERRIDES = {}

# A junction fails when |sum(downstream)-sum(upstream)| is larger than
# ABSOLUTE + RELATIVE * max(|upstream|, |downstream|, MINIMUM_REFERENCE).
MASS_RELATIVE_TOLERANCE = 0.01
MASS_ABSOLUTE_TOLERANCE_CMS = 5.0
MASS_MINIMUM_REFERENCE_FLOW_CMS = 5.0
MAP_DPI = 250

MOI_REPO_OVERRIDE = None
cwd = Path.cwd().resolve()
repo_candidates = [
    MOI_REPO_OVERRIDE,
    cwd,
    cwd.parent,
    Path("/nas/cee-water/cjgleason/Yushan/Confluence_Aug/MOI_gage_correlation"),
    Path("/nas/cee-water/cjgleason/Yushan/Confluence_Aug/confluence_global_v17c_gagecorr/modules/moi"),
]
MOI_REPO = next(
    (Path(path).resolve() for path in repo_candidates if path is not None and (Path(path) / "run_MOI.py").is_file()),
    None,
)
if MOI_REPO is None:
    raise FileNotFoundError("Cannot locate the MOI repository; set MOI_REPO_OVERRIDE.")

ANALYSIS_DIR = MOI_REPO / "analysis"
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

# Pick up Mean-only code changes when this cell is rerun in an existing kernel.
for module_name in ("moi.Integrate", "run_MOI", "three_run_mass_conservation"):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

from three_run_mass_conservation import (
    build_run_specs,
    discover_basin_catalog,
    discover_svs_file,
    load_experiment_results,
    plot_all_basin_maps,
    prepare_all_gage_calval_csv,
    print_junction_mass_summary,
    run_three_experiments,
)

SOURCE_CALVAL_CSV = MOI_REPO / "CalValSeparation_basin_stratified_v2.csv"
SVS_DIR = RUN_ROOT / "input" / "svs"
ALL_GAGE_CALVAL_CSV = RESULT_ROOT / "config" / "CalVal_all_gages_six_basins.csv"

print(f"MOI repository : {MOI_REPO}")
print(f"Unity run root : {RUN_ROOT}")
print(f"Result root    : {RESULT_ROOT}")
print(f"Target basins  : {TARGET_BASINS}")
print(f"SLURM job ID   : {os.environ.get('SLURM_JOB_ID', 'not set')}")


In [ ]:
# Cell 2 — validate the Unity runtime and data paths
required_packages = [
    "numpy", "pandas", "scipy", "netCDF4", "matplotlib",
    "cvxpy", "osqp", "scs",
]
missing_packages = []
for package in required_packages:
    try:
        importlib.import_module(package)
    except ImportError:
        missing_packages.append(package)
if missing_packages:
    raise ImportError(
        "The active kernel is missing: " + ", ".join(missing_packages)
        + f". Install {ANALYSIS_DIR / 'requirements_all_gage_notebook.txt'} into this kernel."
    )

required_paths = [
    RUN_ROOT / "input" / "sos",
    RUN_ROOT / "input" / "swot",
    RUN_ROOT / "input" / "sword",
    RUN_ROOT / "flpe",
    SOURCE_CALVAL_CSV,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required paths:\n" + "\n".join(missing_paths))

SVS_FILE = discover_svs_file(SVS_DIR)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
if os.environ.get("SLURM_JOB_ID") is None:
    warnings.warn(
        "SLURM_JOB_ID is not set. Run the solve cells on a Unity compute allocation."
    )
print(f"SVS file: {SVS_FILE}")
print("Runtime and paths are ready.")


## Prepare the three runs

The run3 CSV is an experiment-only copy. The repository's source Cal/Val file is never modified. `excluded_insufficient_observations` rows in these six basins are also relabeled as calibration, consistent with the requested all-gage capacity test; gages without usable SVS samples are still dropped by the normal input checks.


In [ ]:
# Cell 3 — create run3 CSV and locate the six basin records
from IPython.display import display

calval_audit = prepare_all_gage_calval_csv(
    SOURCE_CALVAL_CSV,
    ALL_GAGE_CALVAL_CSV,
    TARGET_BASINS,
)
calval_audit.to_csv(RESULT_ROOT / "calval_conversion_audit.csv", index=False)
display(calval_audit)

RUN_SPECS = build_run_specs(SOURCE_CALVAL_CSV, ALL_GAGE_CALVAL_CSV)
display(__import__("pandas").DataFrame([
    {
        "run": spec.name,
        "label": spec.label,
        "branch": spec.branch,
        "calval_csv": str(spec.calval_csv) if spec.calval_csv else "none",
    }
    for spec in RUN_SPECS
]))

basin_catalog = discover_basin_catalog(RUN_ROOT / "input")
missing_basins = [basin for basin in TARGET_BASINS if basin not in basin_catalog]
if missing_basins:
    raise KeyError(f"Target basins missing from input JSON files: {missing_basins}")

basin_records = __import__("pandas").DataFrame([
    {
        "basin_id": basin_id,
        "listed_reaches": len(basin_catalog[basin_id]["reach_ids"]),
        "sos": basin_catalog[basin_id]["sos"],
        "sword": basin_catalog[basin_id]["sword"],
        "source_json": basin_catalog[basin_id]["source_json"],
    }
    for basin_id in TARGET_BASINS
])
basin_records.to_csv(RESULT_ROOT / "selected_basin_input_records.csv", index=False)
display(basin_records)


## Run or resume MOI

This cell runs 18 combinations (6 basins × 3 runs), but each combination solves only `Mean` and skips final FLP estimation when `MEAN_ONLY=True`. Each completed combination immediately writes compact MetroMan reach and junction CSVs, so an interrupted kernel can resume with `REUSE_EXISTING=True`. `VERBOSE=False` avoids the very large per-reach solver log seen in yesterday's notebook.


In [ ]:
# Cell 4 — execute or reload all three experiments
if RUN_EXPERIMENTS:
    reach_results, junction_results, mass_summary, run_audit = run_three_experiments(
        TARGET_BASINS,
        basin_catalog,
        RUN_SPECS,
        moi_repo=MOI_REPO,
        run_root=RUN_ROOT,
        svs_file=SVS_FILE,
        result_root=RESULT_ROOT,
        relative_tolerance=MASS_RELATIVE_TOLERANCE,
        absolute_tolerance_cms=MASS_ABSOLUTE_TOLERANCE_CMS,
        minimum_reference_flow_cms=MASS_MINIMUM_REFERENCE_FLOW_CMS,
        parameter_overrides=PARAMETER_OVERRIDES,
        mean_only=MEAN_ONLY,
        verbose=VERBOSE,
        reuse_existing=REUSE_EXISTING,
        continue_on_error=CONTINUE_ON_ERROR,
    )
else:
    reach_results, junction_results, mass_summary, run_audit = load_experiment_results(
        RESULT_ROOT
    )

display(run_audit)
if not run_audit.empty and (run_audit["status"] == "failed").any():
    warnings.warn("At least one run failed; inspect error_log in run_audit.csv.")
if reach_results.empty:
    raise RuntimeError("No MetroMan reach results are available.")


In [ ]:
# Cell 5 — print the fraction of junctions that do not conserve mass
print(
    "Failure definition: abs residual > "
    f"{MASS_ABSOLUTE_TOLERANCE_CMS:g} m3/s + "
    f"{MASS_RELATIVE_TOLERANCE:.2%} of the larger upstream/downstream total."
)
print("The denominator contains only junctions with finite positive qbar on every connected reach.\n")
print_junction_mass_summary(mass_summary)

display_columns = [
    "basin_id", "run", "scale",
    "n_junctions_total", "n_junctions_evaluable",
    "n_junctions_not_conserving", "fraction_junctions_not_conserving",
    "median_relative_residual", "p95_relative_residual",
]
display(mass_summary[display_columns].round(4))


In [ ]:
# Cell 6 — plot qbar_reachScale and qbar_basinScale on river centerlines
import matplotlib.pyplot as plt

figures = plot_all_basin_maps(
    reach_results,
    TARGET_BASINS,
    RUN_SPECS,
    basin_catalog,
    junction_results=junction_results,
    sword_dir=RUN_ROOT / "input" / "sword",
    result_root=RESULT_ROOT,
    dpi=MAP_DPI,
)
plt.show()


In [ ]:
# Cell 7 — output inventory
print(f"All outputs:             {RESULT_ROOT}")
print(f"Run audit:               {RESULT_ROOT / 'run_audit.csv'}")
print(f"All MetroMan qbar:        {RESULT_ROOT / 'all_metroman_qbar.csv'}")
print(f"All junction diagnostics: {RESULT_ROOT / 'all_junction_mass.csv'}")
print(f"Mass summary:             {RESULT_ROOT / 'junction_mass_summary.csv'}")
print(f"Maps:                     {RESULT_ROOT / 'figures'}")
